# Starbucks Online App 분석 통합 프로젝트

## 목표

- 합성 스타벅스 앱 데이터를 생성한다.
- Neon PostgreSQL 에 적재하고 SQL 분석을 수행한다.
- Schema Intelligence 와 Text-to-SQL 품질 개선 흐름을 확인한다.


In [2]:
# 이 통합 노트북에서 사용할 패키지를 설치합니다.
%pip install -q --upgrade \
    "gradio==4.44.1" \
    "pandas==2.2.2" \
    "numpy>=2.0,<2.1" \
    "jedi>=0.19.1" \
    "sqlalchemy>=2.0" psycopg2-binary pandas numpy tabulate sqlparse \
    chromadb llama-index llama-index-llms-openai llama-index-embeddings-openai \
    llama-index-vector-stores-chroma openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB

In [3]:
# ============================================================
# NumPy / pandas ABI 호환성 자동 재시작 (Colab 전용)
# ============================================================
def _need_restart() -> bool:
    import importlib
    import importlib.metadata as _md

    importlib.invalidate_caches()

    try:
        import numpy as _np
        if _md.version("numpy") != _np.__version__:
            return True
    except Exception:
        pass

    try:
        import pandas as _pd
        _ = _pd.Series([1, 2, 3]).dtype
        if _md.version("pandas") != _pd.__version__:
            return True
    except ValueError:
        return True
    except Exception:
        pass

    return False

if _need_restart():
    print("런타임 재시작이 필요합니다. Colab 에서는 Runtime -> Run all 로 다시 실행하세요.")
    try:
        from google.colab import runtime  # type: ignore
        runtime.unassign()
    except Exception:
        raise SystemExit("런타임을 재시작한 뒤 다시 실행해주세요.")
else:
    print("ABI check passed.")


ABI check passed.


In [4]:
# Colab/로컬 환경에서 Secret 을 안전하게 읽습니다.
import os


def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return

    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None

    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None

    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or export it first.")


_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=True)
print("Environment ready.")


Environment ready.


## 프로젝트 개요

이 프로젝트의 역할은 **운영 분석가용 스타벅스 앱 데이터 상담사**입니다.
핵심 질문은 아래와 같습니다.

- 어떤 메뉴가 가장 많이 노출되었는가?
- 노출 대비 클릭(CTR)이 좋은 메뉴는 무엇인가?
- 클릭 대비 주문 전환이 낮은 메뉴는 무엇인가?
- 카테고리별 매출은 어떠한가?
- 재주문 사용자들은 무엇을 자주 구매하는가?

### Mermaid ERD

```mermaid
erDiagram
    USERS ||--o{ VIEW_IMP_LOGS : sees
    USERS ||--o{ CLICK_LOGS : clicks
    USERS ||--o{ ORDERS : places
    MENUS ||--o{ VIEW_IMP_LOGS : appears_in
    MENUS ||--o{ CLICK_LOGS : clicked_for
    MENUS ||--o{ ORDER_ITEMS : ordered_as
    ORDERS ||--|{ ORDER_ITEMS : contains
```


In [5]:
import math
import random
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, inspect, text

SEED = 20260513
random.seed(SEED)
np.random.seed(SEED)

STARBUCKS_TABLES = [
    "users",
    "menus",
    "orders",
    "order_items",
    "view_imp_logs",
    "click_logs",
]

MAX_USERS = 100
N_USERS = 84
N_MENUS = 28
TARGET_IMPRESSIONS = 4200
TARGET_CLICKS = 820
TARGET_ORDERS = 310

assert N_USERS <= MAX_USERS
print("Seed + constants ready.")


Seed + constants ready.


## 합성 데이터 생성

- 사용자 수는 최대 100명으로 고정합니다.
- `users`, `menus`를 먼저 만들고, 그 다음 `view_imp_logs`, `click_logs`, `orders`, `order_items`를 생성합니다.
- 일부 사용자는 의도적으로 **재주문 성향**을 높여 재방문/재구매 분석이 가능하도록 만듭니다.
- `orders.total_amount`는 항상 `order_items.line_amount` 합계와 같게 만듭니다.


In [6]:
BASE_TS = datetime(2026, 4, 1, 9, 0, 0)

membership_tiers = ["Welcome", "Green", "Gold"]
age_bands = ["20s", "30s", "40s", "50s"]
genders = ["F", "M"]
channels = ["app", "web"]
surfaces = ["home_banner", "recommendation", "category_tab", "search_result"]
devices = ["ios", "android", "web"]
store_types = ["pickup", "delivery"]
order_statuses = ["completed", "completed", "completed", "cancelled"]
menu_catalog = [
    ("Caffe Americano", "Coffee", "Brewed", "hot", 4500, False, False),
    ("Iced Caffe Americano", "Coffee", "Brewed", "iced", 4500, False, False),
    ("Caffe Latte", "Coffee", "Latte", "hot", 5000, False, False),
    ("Iced Caffe Latte", "Coffee", "Latte", "iced", 5000, False, False),
    ("Caramel Macchiato", "Coffee", "Latte", "hot", 5900, False, False),
    ("Iced Caramel Macchiato", "Coffee", "Latte", "iced", 5900, False, False),
    ("Cold Brew", "Coffee", "Cold Brew", "iced", 4900, False, False),
    ("Vanilla Cream Cold Brew", "Coffee", "Cold Brew", "iced", 5800, False, False),
    ("Strawberry Acai Lemonade", "Refreshers", "Refresher", "iced", 6100, True, True),
    ("Mango Dragonfruit Lemonade", "Refreshers", "Refresher", "iced", 6100, True, True),
    ("Green Tea Latte", "Tea", "Latte", "hot", 5700, False, False),
    ("Iced Green Tea Latte", "Tea", "Latte", "iced", 5700, False, False),
    ("Signature Chocolate", "Tea", "Chocolate", "hot", 5900, False, False),
    ("Iced Signature Chocolate", "Tea", "Chocolate", "iced", 5900, False, False),
    ("Blueberry Bagel", "Food", "Bakery", "none", 3900, False, False),
    ("Butter Croissant", "Food", "Bakery", "none", 4200, False, False),
    ("Ham Cheese Sandwich", "Food", "Sandwich", "none", 6200, False, False),
    ("Chicken Pesto Sandwich", "Food", "Sandwich", "none", 6500, False, False),
    ("Tiramisu Cake", "Dessert", "Cake", "none", 6900, False, False),
    ("Chocolate Muffin", "Dessert", "Bakery", "none", 4300, False, False),
    ("Jeju Hallabong Ade", "Refreshers", "Ade", "iced", 6500, True, True),
    ("Dolce Cold Brew", "Coffee", "Cold Brew", "iced", 6000, False, False),
    ("Flat White", "Coffee", "Latte", "hot", 5600, False, False),
    ("Yogurt Blended", "Frappuccino", "Blended", "iced", 6300, True, False),
    ("Java Chip Frappuccino", "Frappuccino", "Blended", "iced", 6500, False, False),
    ("Black Tea Lemonade", "Tea", "Tea", "iced", 5400, False, False),
    ("Earl Grey Tea", "Tea", "Tea", "hot", 4900, False, False),
    ("Seasonal Berry Cake", "Dessert", "Cake", "none", 7100, True, True),
]

users = []
for user_id in range(1, N_USERS + 1):
    tier = random.choices(membership_tiers, weights=[0.45, 0.35, 0.20], k=1)[0]
    users.append(
        {
            "user_id": user_id,
            "user_name": f"user_{user_id:03d}",
            "age_band": random.choice(age_bands),
            "gender": random.choice(genders),
            "membership_tier": tier,
            "preferred_channel": random.choice(channels),
            "signup_month": f"2025-{random.randint(1, 12):02d}",
            "created_at": BASE_TS - timedelta(days=random.randint(10, 360)),
        }
    )
users_df = pd.DataFrame(users)

menus = []
for menu_id, (name, category, subcategory, temperature, price, is_seasonal, is_limited) in enumerate(menu_catalog[:N_MENUS], start=1):
    menus.append(
        {
            "menu_id": menu_id,
            "menu_name": name,
            "category": category,
            "subcategory": subcategory,
            "temperature": temperature,
            "price": price,
            "is_seasonal": is_seasonal,
            "is_limited": is_limited,
        }
    )
menus_df = pd.DataFrame(menus)

users_df.head(), menus_df.head()


(   user_id user_name age_band gender membership_tier preferred_channel  \
 0        1  user_001      50s      M         Welcome               app   
 1        2  user_002      30s      F         Welcome               app   
 2        3  user_003      40s      M         Welcome               app   
 3        4  user_004      20s      F           Green               app   
 4        5  user_005      50s      M         Welcome               web   
 
   signup_month          created_at  
 0      2025-01 2025-04-19 09:00:00  
 1      2025-07 2025-11-24 09:00:00  
 2      2025-03 2025-11-06 09:00:00  
 3      2025-08 2025-07-01 09:00:00  
 4      2025-04 2025-07-15 09:00:00  ,
    menu_id             menu_name category subcategory temperature  price  \
 0        1       Caffe Americano   Coffee      Brewed         hot   4500   
 1        2  Iced Caffe Americano   Coffee      Brewed        iced   4500   
 2        3           Caffe Latte   Coffee       Latte         hot   5000   
 3        4

In [7]:
def weighted_menu_choice() -> int:
    menu_weights = []
    for _, row in menus_df.iterrows():
        base = 1.0
        if row["category"] == "Coffee":
            base *= 1.6
        if row["category"] == "Food":
            base *= 0.9
        if row["is_seasonal"]:
            base *= 1.15
        if row["temperature"] == "iced":
            base *= 1.10
        menu_weights.append(base)
    return random.choices(menus_df["menu_id"].tolist(), weights=menu_weights, k=1)[0]


def random_session_id(user_id: int, event_idx: int) -> str:
    return f"s-{user_id:03d}-{event_idx:05d}"


impressions = []
clicks = []
view_id = 1
click_id = 1

for idx in range(TARGET_IMPRESSIONS):
    user_id = random.randint(1, N_USERS)
    menu_id = weighted_menu_choice()
    event_ts = BASE_TS + timedelta(minutes=random.randint(0, 60 * 24 * 45))
    device = random.choices(devices, weights=[0.42, 0.43, 0.15], k=1)[0]
    surface = random.choices(surfaces, weights=[0.35, 0.35, 0.20, 0.10], k=1)[0]
    session_id = random_session_id(user_id, idx)
    impressions.append(
        {
            "view_id": view_id,
            "user_id": user_id,
            "menu_id": menu_id,
            "session_id": session_id,
            "event_ts": event_ts,
            "surface": surface,
            "device_type": device,
            "position_rank": random.randint(1, 12),
        }
    )
    view_id += 1

    click_prob = 0.17
    if surface == "recommendation":
        click_prob += 0.04
    if device == "ios":
        click_prob += 0.01
    if menus_df.loc[menus_df["menu_id"] == menu_id, "is_seasonal"].iloc[0]:
        click_prob += 0.03

    if random.random() < min(click_prob, 0.45):
        clicks.append(
            {
                "click_id": click_id,
                "user_id": user_id,
                "menu_id": menu_id,
                "session_id": session_id,
                "event_ts": event_ts + timedelta(seconds=random.randint(5, 180)),
                "surface": surface,
                "device_type": device,
                "action_type": random.choice(["menu_detail", "add_to_cart", "recommendation_click"]),
            }
        )
        click_id += 1

clicks_df = pd.DataFrame(clicks).head(TARGET_CLICKS).copy()
impressions_df = pd.DataFrame(impressions)

heavy_repeat_users = set(random.sample(users_df["user_id"].tolist(), 14))
orders = []
order_items = []
order_id = 1
order_item_id = 1

click_rows = clicks_df.to_dict("records")
random.shuffle(click_rows)
for row in click_rows:
    user_id = row["user_id"]
    convert_prob = 0.30 if user_id in heavy_repeat_users else 0.18
    if row["action_type"] == "add_to_cart":
        convert_prob += 0.08
    if random.random() >= min(convert_prob, 0.72):
        continue

    status = random.choices(order_statuses, weights=[0.78, 0.10, 0.07, 0.05], k=1)[0]
    order_ts = row["event_ts"] + timedelta(minutes=random.randint(1, 45))
    num_items = random.choices([1, 2, 3], weights=[0.60, 0.30, 0.10], k=1)[0]
    chosen_menu_ids = [row["menu_id"]]
    while len(chosen_menu_ids) < num_items:
        candidate = weighted_menu_choice()
        if candidate not in chosen_menu_ids:
            chosen_menu_ids.append(candidate)

    total_amount = 0
    current_items = []
    for menu_id in chosen_menu_ids:
        unit_price = float(menus_df.loc[menus_df["menu_id"] == menu_id, "price"].iloc[0])
        qty = random.choices([1, 2], weights=[0.82, 0.18], k=1)[0]
        line_amount = unit_price * qty
        total_amount += line_amount
        current_items.append(
            {
                "order_item_id": order_item_id,
                "order_id": order_id,
                "menu_id": menu_id,
                "quantity": qty,
                "unit_price": unit_price,
                "line_amount": line_amount,
            }
        )
        order_item_id += 1

    orders.append(
        {
            "order_id": order_id,
            "user_id": user_id,
            "session_id": row["session_id"],
            "order_ts": order_ts,
            "order_status": status,
            "channel": "app" if row["device_type"] in {"ios", "android"} else "web",
            "store_type": random.choices(store_types, weights=[0.58, 0.42], k=1)[0],
            "total_amount": total_amount,
        }
    )
    order_items.extend(current_items)
    order_id += 1

orders_df = pd.DataFrame(orders).head(TARGET_ORDERS).copy()
valid_order_ids = set(orders_df["order_id"].tolist())
order_items_df = pd.DataFrame([r for r in order_items if r["order_id"] in valid_order_ids]).copy()

# total_amount 를 order_items 기준으로 다시 맞춰 정합성을 보장합니다.
order_totals = order_items_df.groupby("order_id", as_index=False)["line_amount"].sum().rename(columns={"line_amount": "recalc_total"})
orders_df = orders_df.drop(columns=["total_amount"]).merge(order_totals, on="order_id", how="left").rename(columns={"recalc_total": "total_amount"})

print({
    "users": len(users_df),
    "menus": len(menus_df),
    "view_imp_logs": len(impressions_df),
    "click_logs": len(clicks_df),
    "orders": len(orders_df),
    "order_items": len(order_items_df),
})


{'users': 84, 'menus': 28, 'view_imp_logs': 4200, 'click_logs': 820, 'orders': 192, 'order_items': 291}


In [8]:
assert len(users_df) <= 100
assert orders_df["order_id"].is_unique
assert order_items_df["order_item_id"].is_unique
assert set(clicks_df["user_id"]).issubset(set(users_df["user_id"]))
assert set(impressions_df["menu_id"]).issubset(set(menus_df["menu_id"]))
assert set(order_items_df["menu_id"]).issubset(set(menus_df["menu_id"]))

merged_totals = orders_df.merge(
    order_items_df.groupby("order_id", as_index=False)["line_amount"].sum(),
    on="order_id",
    how="left",
)
assert np.allclose(merged_totals["total_amount"], merged_totals["line_amount"])

print("Synthetic data integrity checks passed.")
print(users_df.head(3))
print(menus_df.head(3))
print(impressions_df.head(3))
print(clicks_df.head(3))
print(orders_df.head(3))
print(order_items_df.head(3))


Synthetic data integrity checks passed.
   user_id user_name age_band gender membership_tier preferred_channel  \
0        1  user_001      50s      M         Welcome               app   
1        2  user_002      30s      F         Welcome               app   
2        3  user_003      40s      M         Welcome               app   

  signup_month          created_at  
0      2025-01 2025-04-19 09:00:00  
1      2025-07 2025-11-24 09:00:00  
2      2025-03 2025-11-06 09:00:00  
   menu_id             menu_name category subcategory temperature  price  \
0        1       Caffe Americano   Coffee      Brewed         hot   4500   
1        2  Iced Caffe Americano   Coffee      Brewed        iced   4500   
2        3           Caffe Latte   Coffee       Latte         hot   5000   

   is_seasonal  is_limited  
0        False       False  
1        False       False  
2        False       False  
   view_id  user_id  menu_id   session_id            event_ts         surface  \
0        1   

In [9]:
engine = create_engine(os.environ["NEON_DSN"])


def run_query(sql: str, limit: int | None = None) -> pd.DataFrame:
    clean_sql = sql.strip().rstrip(";")
    if limit is not None:
        clean_sql += f"\nLIMIT {limit}"
    with engine.connect() as conn:
        return pd.read_sql(text(clean_sql), conn)


print("SQLAlchemy engine ready.")


SQLAlchemy engine ready.


In [10]:
DDL_SQL = """
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    user_name TEXT NOT NULL,
    age_band TEXT NOT NULL,
    gender TEXT NOT NULL,
    membership_tier TEXT NOT NULL,
    preferred_channel TEXT NOT NULL,
    signup_month TEXT NOT NULL,
    created_at TIMESTAMP NOT NULL
);

CREATE TABLE IF NOT EXISTS menus (
    menu_id INTEGER PRIMARY KEY,
    menu_name TEXT NOT NULL,
    category TEXT NOT NULL,
    subcategory TEXT NOT NULL,
    temperature TEXT NOT NULL,
    price NUMERIC(10, 2) NOT NULL,
    is_seasonal BOOLEAN NOT NULL,
    is_limited BOOLEAN NOT NULL
);

CREATE TABLE IF NOT EXISTS view_imp_logs (
    view_id BIGINT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(user_id),
    menu_id INTEGER NOT NULL REFERENCES menus(menu_id),
    session_id TEXT NOT NULL,
    event_ts TIMESTAMP NOT NULL,
    surface TEXT NOT NULL,
    device_type TEXT NOT NULL,
    position_rank INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS click_logs (
    click_id BIGINT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(user_id),
    menu_id INTEGER NOT NULL REFERENCES menus(menu_id),
    session_id TEXT NOT NULL,
    event_ts TIMESTAMP NOT NULL,
    surface TEXT NOT NULL,
    device_type TEXT NOT NULL,
    action_type TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS orders (
    order_id BIGINT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(user_id),
    session_id TEXT NOT NULL,
    order_ts TIMESTAMP NOT NULL,
    order_status TEXT NOT NULL,
    channel TEXT NOT NULL,
    store_type TEXT NOT NULL,
    total_amount NUMERIC(12, 2) NOT NULL
);

CREATE TABLE IF NOT EXISTS order_items (
    order_item_id BIGINT PRIMARY KEY,
    order_id BIGINT NOT NULL REFERENCES orders(order_id),
    menu_id INTEGER NOT NULL REFERENCES menus(menu_id),
    quantity INTEGER NOT NULL,
    unit_price NUMERIC(10, 2) NOT NULL,
    line_amount NUMERIC(12, 2) NOT NULL
);
"""

COMMENT_SQL = """
COMMENT ON TABLE users IS '스타벅스 온라인 앱 사용자 마스터';
COMMENT ON COLUMN users.membership_tier IS 'Welcome, Green, Gold 멤버십 등급';
COMMENT ON TABLE menus IS '스타벅스 메뉴 마스터';
COMMENT ON COLUMN menus.category IS 'Coffee, Food, Tea 같은 상위 카테고리';
COMMENT ON TABLE view_imp_logs IS '앱 화면 메뉴 노출 로그';
COMMENT ON COLUMN view_imp_logs.surface IS '노출 지면: home_banner, recommendation, category_tab, search_result';
COMMENT ON TABLE click_logs IS '앱 화면 메뉴 클릭 로그';
COMMENT ON COLUMN click_logs.action_type IS 'menu_detail, add_to_cart, recommendation_click 같은 액션';
COMMENT ON TABLE orders IS '주문 헤더. completed 만 매출 집계의 기본 기준';
COMMENT ON COLUMN orders.order_status IS 'completed, cancelled 등 주문 상태';
COMMENT ON TABLE order_items IS '주문 상세 라인. 주문별 메뉴 구성';
COMMENT ON COLUMN order_items.line_amount IS 'quantity * unit_price';
"""

RESET_SQL = """
TRUNCATE TABLE order_items, orders, click_logs, view_imp_logs, menus, users RESTART IDENTITY CASCADE;
"""

with engine.begin() as conn:
    conn.execute(text(DDL_SQL))
    conn.execute(text(RESET_SQL))

users_df.to_sql("users", engine, if_exists="append", index=False)
menus_df.to_sql("menus", engine, if_exists="append", index=False)
impressions_df.to_sql("view_imp_logs", engine, if_exists="append", index=False)
clicks_df.to_sql("click_logs", engine, if_exists="append", index=False)
orders_df.to_sql("orders", engine, if_exists="append", index=False)
order_items_df.to_sql("order_items", engine, if_exists="append", index=False)

with engine.begin() as conn:
    conn.execute(text(COMMENT_SQL))

print("Neon table creation + load complete.")


Neon table creation + load complete.


In [11]:
for table_name in STARBUCKS_TABLES:
    df_count = run_query(f"SELECT COUNT(*) AS cnt FROM {table_name}")
    print(f"{table_name}: {int(df_count['cnt'].iloc[0])}")

run_query(
    """
    SELECT o.order_id, u.user_name, m.menu_name, oi.quantity, oi.line_amount
    FROM orders o
    JOIN users u ON u.user_id = o.user_id
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN menus m ON m.menu_id = oi.menu_id
    ORDER BY o.order_ts DESC
    LIMIT 10
    """
)


users: 84
menus: 28
orders: 192
order_items: 291
view_imp_logs: 4200
click_logs: 820


,order_id,user_name,menu_name,quantity,line_amount
0,135,user_026,Java Chip Frappuccino,1,6500.0
1,14,user_043,Tiramisu Cake,1,6900.0
2,14,user_043,Dolce Cold Brew,1,6000.0
3,14,user_043,Ham Cheese Sandwich,1,6200.0
4,32,user_013,Butter Croissant,1,4200.0
5,79,user_010,Iced Caramel Macchiato,1,5900.0
6,73,user_018,Caramel Macchiato,1,5900.0
7,52,user_062,Vanilla Cream Cold Brew,1,5800.0
8,52,user_062,Iced Caffe Americano,1,4500.0
9,52,user_062,Caramel Macchiato,1,5900.0


## PostgreSQL로 데이터 사전 확인

기본 조회와 실행 계획을 스타벅스 도메인으로 확인합니다.


In [12]:
print(run_query("SELECT user_id, user_name, membership_tier FROM users ORDER BY user_id LIMIT 5"))
print(run_query("SELECT menu_name, category, price FROM menus ORDER BY price DESC LIMIT 5"))
print(run_query("SELECT order_id, order_status, total_amount FROM orders ORDER BY total_amount DESC LIMIT 5"))


   user_id user_name membership_tier
0        1  user_001         Welcome
1        2  user_002         Welcome
2        3  user_003         Welcome
3        4  user_004           Green
4        5  user_005         Welcome
                menu_name     category   price
0     Seasonal Berry Cake      Dessert  7100.0
1           Tiramisu Cake      Dessert  6900.0
2      Jeju Hallabong Ade   Refreshers  6500.0
3  Chicken Pesto Sandwich         Food  6500.0
4   Java Chip Frappuccino  Frappuccino  6500.0
   order_id order_status  total_amount
0        74    completed       25200.0
1       116    completed       24800.0
2         1    completed       23700.0
3       140    completed       23100.0
4       179    completed       22300.0


In [13]:
explain_df = run_query(
    """
    EXPLAIN
    SELECT m.category, ROUND(SUM(o.total_amount), 0) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN menus m ON m.menu_id = oi.menu_id
    WHERE o.order_status = 'completed'
    GROUP BY m.category
    ORDER BY revenue DESC
    """
)
explain_df


,QUERY PLAN
0,Sort (cost=38.84..38.86 rows=5 width=64)
1,"Sort Key: (round(sum(o.total_amount), 0)) DESC"
2,-> GroupAggregate (cost=38.67..38.79 rows=...
3,Group Key: m.category
4,-> Sort (cost=38.67..38.69 rows=5 wi...
5,Sort Key: m.category
6,-> Nested Loop (cost=15.42..38...
7,-> Hash Join (cost=15.28...
8,Hash Cond: (oi.order...
9,-> Seq Scan on orde...



## 고급 SQL로 데이터 사전 확인

 `GROUP BY`, `JOIN`, `CTE`, `윈도우 함수`로 스타벅스 데이터를 확인합니다.


In [14]:
queries = {
    "카테고리별 매출": """
        SELECT m.category, ROUND(SUM(oi.line_amount), 0) AS revenue
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        JOIN menus m ON m.menu_id = oi.menu_id
        WHERE o.order_status = 'completed'
        GROUP BY m.category
        ORDER BY revenue DESC
    """,
    "메뉴별 CTR": """
        WITH imp AS (
            SELECT menu_id, COUNT(*) AS impressions
            FROM view_imp_logs
            GROUP BY menu_id
        ), clk AS (
            SELECT menu_id, COUNT(*) AS clicks
            FROM click_logs
            GROUP BY menu_id
        )
        SELECT m.menu_name,
               COALESCE(i.impressions, 0) AS impressions,
               COALESCE(c.clicks, 0) AS clicks,
               ROUND(COALESCE(c.clicks, 0)::numeric / NULLIF(i.impressions, 0), 4) AS ctr
        FROM menus m
        LEFT JOIN imp i ON i.menu_id = m.menu_id
        LEFT JOIN clk c ON c.menu_id = m.menu_id
        WHERE COALESCE(i.impressions, 0) > 0
        ORDER BY ctr DESC NULLS LAST, impressions DESC
        LIMIT 10
    """,
    "사용자별 주문 빈도": """
        SELECT u.user_name,
               COUNT(*) AS completed_orders,
               ROUND(AVG(o.total_amount), 0) AS avg_order_amount
        FROM users u
        JOIN orders o ON o.user_id = u.user_id
        WHERE o.order_status = 'completed'
        GROUP BY u.user_name
        ORDER BY completed_orders DESC, avg_order_amount DESC
        LIMIT 10
    """,
    "최근 30일 Top 메뉴": """
        SELECT m.menu_name,
               SUM(oi.quantity) AS qty,
               ROUND(SUM(oi.line_amount), 0) AS revenue,
               RANK() OVER (ORDER BY SUM(oi.line_amount) DESC) AS revenue_rank
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        JOIN menus m ON m.menu_id = oi.menu_id
        WHERE o.order_status = 'completed'
          AND o.order_ts >= (SELECT MAX(order_ts) - INTERVAL '30 days' FROM orders)
        GROUP BY m.menu_name
        ORDER BY revenue_rank
        LIMIT 10
    """,
}

for label, sql in queries.items():
    print(f"\n=== {label} ===")
    display(run_query(sql))



=== 카테고리별 매출 ===


,category,revenue
0,Coffee,785400.0
1,Tea,353300.0
2,Refreshers,188600.0
3,Dessert,172000.0
4,Food,128700.0
5,Frappuccino,115000.0



=== 메뉴별 CTR ===


,menu_name,impressions,clicks,ctr
0,Flat White,204,52,0.2549
1,Earl Grey Tea,107,26,0.2430
2,Seasonal Berry Cake,145,35,0.2414
3,Iced Caramel Macchiato,175,42,0.2400
4,Iced Green Tea Latte,114,27,0.2368
5,Mango Dragonfruit Lemonade,157,37,0.2357
6,Iced Caffe Americano,198,44,0.2222
7,Yogurt Blended,137,30,0.2190
8,Strawberry Acai Lemonade,119,26,0.2185
9,Signature Chocolate,128,27,0.2109



=== 사용자별 주문 빈도 ===


,user_name,completed_orders,avg_order_amount
0,user_060,9,9311.0
1,user_045,5,15020.0
2,user_074,5,9080.0
3,user_031,4,13625.0
4,user_080,4,13250.0
5,user_061,4,13100.0
6,user_043,4,13050.0
7,user_010,4,10825.0
8,user_052,4,9700.0
9,user_065,4,9150.0



=== 최근 30일 Top 메뉴 ===


,menu_name,qty,revenue,revenue_rank
0,Dolce Cold Brew,17,102000.0,1
1,Iced Caffe Americano,16,72000.0,2
2,Caramel Macchiato,12,70800.0,3
3,Jeju Hallabong Ade,10,65000.0,4
4,Seasonal Berry Cake,9,63900.0,5
5,Flat White,11,61600.0,6
6,Iced Green Tea Latte,10,57000.0,7
7,Tiramisu Cake,6,41400.0,8
8,Iced Signature Chocolate,7,41300.0,9
9,Java Chip Frappuccino,6,39000.0,10


## Schema Intelligence

LLM 은 스키마를 결국 **텍스트**로 읽습니다.
좋은 네이밍, FK, `COMMENT ON`, 화이트리스트가 Text-to-SQL 품질에 직접 영향을 줍니다.


In [15]:
inspector = inspect(engine)
print("Tables:", inspector.get_table_names())
for table_name in STARBUCKS_TABLES:
    print(f"\n--- {table_name} columns ---")
    for col in inspector.get_columns(table_name):
        print(f"{col['name']:<18} {str(col['type']):<20} nullable={col['nullable']}")

comments_df = run_query(
    """
    SELECT c.relname AS table_name,
           a.attname AS column_name,
           d.description AS comment
    FROM pg_class c
    JOIN pg_attribute a ON a.attrelid = c.oid AND a.attnum > 0 AND NOT a.attisdropped
    LEFT JOIN pg_description d ON d.objoid = c.oid AND d.objsubid = a.attnum
    WHERE c.relname IN ('users', 'menus', 'view_imp_logs', 'click_logs', 'orders', 'order_items')
    ORDER BY c.relname, a.attnum
    """
)
comments_df.head(20)


Tables: ['departments', 'doctors', 'patients', 'visits', 'diagnoses', 'users', 'view_imp_logs', 'menus', 'click_logs', 'orders', 'order_items']

--- users columns ---
user_id            INTEGER              nullable=False
user_name          TEXT                 nullable=False
age_band           TEXT                 nullable=False
gender             TEXT                 nullable=False
membership_tier    TEXT                 nullable=False
preferred_channel  TEXT                 nullable=False
signup_month       TEXT                 nullable=False
created_at         TIMESTAMP            nullable=False

--- menus columns ---
menu_id            INTEGER              nullable=False
menu_name          TEXT                 nullable=False
category           TEXT                 nullable=False
subcategory        TEXT                 nullable=False
temperature        TEXT                 nullable=False
price              NUMERIC(10, 2)       nullable=False
is_seasonal        BOOLEAN              

,table_name,column_name,comment
0,click_logs,click_id,None
1,click_logs,user_id,None
2,click_logs,menu_id,None
3,click_logs,session_id,None
4,click_logs,event_ts,None
5,click_logs,surface,None
6,click_logs,device_type,None
7,click_logs,action_type,"menu_detail, add_to_cart, recommendation_click..."
8,menus,menu_id,None
9,menus,menu_name,None


In [16]:
from llama_index.core import SQLDatabase

sql_db_schema_only = SQLDatabase(engine, include_tables=STARBUCKS_TABLES)
print(sql_db_schema_only.get_single_table_info("orders"))
print("\n" + "=" * 80 + "\n")
print(sql_db_schema_only.get_single_table_info("view_imp_logs"))


Table 'orders' has columns: order_id (BIGINT), user_id (INTEGER), session_id (TEXT), order_ts (TIMESTAMP), order_status (TEXT): 'completed, cancelled 등 주문 상태', channel (TEXT), store_type (TEXT), total_amount (NUMERIC(12, 2)), with comment: (주문 헤더. completed 만 매출 집계의 기본 기준)  and foreign keys: ['user_id'] -> users.['user_id'].


Table 'view_imp_logs' has columns: view_id (BIGINT), user_id (INTEGER), menu_id (INTEGER), session_id (TEXT), event_ts (TIMESTAMP), surface (TEXT): '노출 지면: home_banner, recommendation, category_tab, search_result', device_type (TEXT), position_rank (INTEGER), with comment: (앱 화면 메뉴 노출 로그)  and foreign keys: ['menu_id'] -> menus.['menu_id'], ['user_id'] -> users.['user_id'].


## LlamaIndex / Embedding / Chroma

짧은 스타벅스 도메인 문서를 만들고, `Document -> chunk -> index -> query engine` 흐름을 확인합니다.


In [17]:
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

starbucks_docs = [
    Document(text="""
    스타벅스 온라인 앱의 핵심 퍼널은 노출 -> 클릭 -> 주문입니다.
    운영 분석에서는 단순 클릭 수보다 CTR 과 클릭 대비 주문 전환율을 함께 봅니다.
    """),
    Document(text="""
    orders.order_status 가 completed 인 주문만 기본 매출 집계 대상으로 사용합니다.
    cancelled 주문은 실적 집계에서 제외합니다.
    """),
    Document(text="""
    메뉴 성과 분석은 category, subcategory, is_seasonal, device_type, surface 를 함께 봐야 합니다.
    recommendation 지면과 home_banner 는 CTR 차이가 날 수 있습니다.
    """),
    Document(text="""
    재주문 사용자 분석은 동일 user_id 의 completed 주문이 2건 이상인 경우를 기본 정의로 사용합니다.
    """),
]

splitter = SentenceSplitter(chunk_size=256, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents(starbucks_docs)
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=3)
retriever = index.as_retriever(similarity_top_k=2)

print(query_engine.query("이 데이터셋에서 매출 집계에 어떤 주문 상태를 써야 하나요?"))
retrieved = retriever.retrieve("CTR 분석에서 같이 봐야 하는 차원은 무엇인가요?")
print([n.node.text[:120] for n in retrieved])


매출 집계에는 주문 상태가 'completed'인 주문만 사용해야 합니다.
['메뉴 성과 분석은 category, subcategory, is_seasonal, device_type, surface 를 함께 봐야 합니다.\n    recommendation 지면과 home_banner 는 CTR', '스타벅스 온라인 앱의 핵심 퍼널은 노출 -> 클릭 -> 주문입니다.\n    운영 분석에서는 단순 클릭 수보다 CTR 과 클릭 대비 주문 전환율을 함께 봅니다.']


In [18]:
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

persist_dir = Path("./starbucks_chroma")
persist_dir.mkdir(exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(persist_dir))
collection = chroma_client.get_or_create_collection("starbucks_app_docs")
vector_store = ChromaVectorStore(chroma_collection=collection)
chroma_index = VectorStoreIndex(nodes, vector_store=vector_store)
print("Chroma persistence ready at:", persist_dir.resolve())
print(chroma_index.as_query_engine(similarity_top_k=2).query("재주문 사용자의 기본 정의는 무엇인가요?"))


Chroma persistence ready at: /content/starbucks_chroma
재주문 사용자의 기본 정의는 동일 user_id의 completed 주문이 2건 이상인 경우입니다.


## Text-to-SQL 기본

이제 LlamaIndex 의 `SQLDatabase + NLSQLTableQueryEngine` 로 자연어 질문을 SQL 로 바꿔봅니다.


In [19]:
from llama_index.core.query_engine import NLSQLTableQueryEngine

sql_db = SQLDatabase(engine, include_tables=STARBUCKS_TABLES)

nlq_basic = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=STARBUCKS_TABLES,
)
print("Basic NLSQLTableQueryEngine ready.")


Basic NLSQLTableQueryEngine ready.


In [20]:
basic_examples = [
    "전체 사용자 수는 몇 명인가요?",
    "완료된 주문 수는 몇 건인가요?",
    "메뉴 카테고리 목록을 보여주세요.",
]

for question in basic_examples:
    resp = nlq_basic.query(question)
    print("\nQ:", question)
    print("SQL:", resp.metadata.get("sql_query", ""))
    print("A:", str(resp.response)[:300])



Q: 전체 사용자 수는 몇 명인가요?
SQL: SELECT COUNT(user_id) AS total_users FROM users;
A: 전체 사용자 수는 84명입니다.

Q: 완료된 주문 수는 몇 건인가요?
SQL: SELECT COUNT(order_id) AS completed_orders FROM orders WHERE order_status = 'completed';
A: 완료된 주문 수는 173건입니다.

Q: 메뉴 카테고리 목록을 보여주세요.
SQL: SELECT DISTINCT category FROM menus ORDER BY category;
A: 메뉴 카테고리 목록은 다음과 같습니다:

- 커피
- 디저트
- 음식
- 프라푸치노
- 리프레셔
- 차


## Text-to-SQL 심화

기본 엔진은 스키마는 보지만, 실무 규칙까지는 잘 모를 수 있습니다.
따라서 아래 세 가지를 추가합니다.

1. 도메인 규칙 주입
2. few-shot 예시 주입
3. 관련 테이블 자동 선택


In [21]:
from llama_index.core.prompts import PromptTemplate

DOMAIN_CONTEXT = """
## 도메인 설명
이 데이터베이스는 스타벅스 온라인 앱 분석용입니다.

## 핵심 규칙
- orders.order_status = 'completed' 인 주문만 기본 매출 집계 대상으로 사용합니다.
- CTR = clicks / impressions 입니다.
- 주문 전환은 보통 click_logs 와 orders 를 user_id + session_id 기준으로 연결해 해석합니다.
- '최근'은 기본적으로 데이터셋의 최대 날짜 기준 최근 30일로 해석합니다.
- '재주문 사용자'는 completed 주문이 2건 이상인 사용자입니다.
- 메뉴 성과는 menu_name 뿐 아니라 category, device_type, surface 도 함께 해석할 수 있습니다.
"""

custom_text_to_sql_prompt = PromptTemplate(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마와 규칙을 참고하여 SQL 을 작성하세요.

## 데이터베이스 스키마
{schema}

"""
    + DOMAIN_CONTEXT
    +
    """
## 예시
질문: 전체 사용자 수는 몇 명인가요?
SQL: SELECT COUNT(*) AS total_users FROM users;

질문: 카테고리별 매출은?
SQL: SELECT m.category, ROUND(SUM(oi.line_amount), 0) AS revenue
     FROM orders o
     JOIN order_items oi ON oi.order_id = o.order_id
     JOIN menus m ON m.menu_id = oi.menu_id
     WHERE o.order_status = 'completed'
     GROUP BY m.category
     ORDER BY revenue DESC;

질문: 메뉴별 CTR 상위 5개는?
SQL: WITH imp AS (
         SELECT menu_id, COUNT(*) AS impressions
         FROM view_imp_logs
         GROUP BY menu_id
     ), clk AS (
         SELECT menu_id, COUNT(*) AS clicks
         FROM click_logs
         GROUP BY menu_id
     )
     SELECT m.menu_name,
            COALESCE(c.clicks, 0) AS clicks,
            COALESCE(i.impressions, 0) AS impressions,
            ROUND(COALESCE(c.clicks, 0)::numeric / NULLIF(i.impressions, 0), 4) AS ctr
     FROM menus m
     LEFT JOIN imp i ON i.menu_id = m.menu_id
     LEFT JOIN clk c ON c.menu_id = m.menu_id
     WHERE COALESCE(i.impressions, 0) > 0
     ORDER BY ctr DESC NULLS LAST
     LIMIT 5;

질문: {query_str}
SQL:"""
)
print("Custom prompt ready.")


Custom prompt ready.


In [22]:
from llama_index.core.objects import ObjectIndex, SQLTableNodeMapping, SQLTableSchema

try:
    from llama_index.core import VectorStoreIndex as _ObjectVectorStoreIndex
except ImportError:
    from llama_index.core.indices.vector_store import VectorStoreIndex as _ObjectVectorStoreIndex

node_mapping = SQLTableNodeMapping(sql_db)
table_schemas = [
    SQLTableSchema(table_name="users", context_str="앱 사용자 마스터. 멤버십 등급, 유입 채널, 가입 시점."),
    SQLTableSchema(table_name="menus", context_str="메뉴 마스터. 카테고리, 서브카테고리, 가격, 시즌 여부."),
    SQLTableSchema(table_name="orders", context_str="주문 헤더. completed 상태만 매출 집계 기본 기준."),
    SQLTableSchema(table_name="order_items", context_str="주문 상세 라인. 어떤 메뉴가 몇 개 팔렸는지 저장."),
    SQLTableSchema(table_name="view_imp_logs", context_str="메뉴 노출 로그. surface 와 device_type 으로 CTR 분석."),
    SQLTableSchema(table_name="click_logs", context_str="메뉴 클릭 로그. user_id + session_id 로 주문 전환과 연결 가능."),
]

obj_index = ObjectIndex.from_objects(
    table_schemas,
    node_mapping,
    _ObjectVectorStoreIndex,
)
print("ObjectIndex ready.")


ObjectIndex ready.


In [30]:
from llama_index.core.indices.struct_store.sql_query import SQLTableRetrieverQueryEngine

enhanced_engine = NLSQLTableQueryEngine(
    sql_database=sql_db,
    tables=STARBUCKS_TABLES,
    text_to_sql_prompt=custom_text_to_sql_prompt
)
auto_engine = SQLTableRetrieverQueryEngine(
    sql_database=sql_db,
    table_retriever=obj_index.as_retriever(similarity_top_k=3),
    text_to_sql_prompt=custom_text_to_sql_prompt,
)
print("Enhanced prompt engine ready.")
print("Enhanced Text-to-SQL engine ready.")


Enhanced prompt engine ready.
Enhanced Text-to-SQL engine ready.


In [31]:
QUESTION_CASES = [
    {
        "difficulty": "기초",
        "question": "전체 사용자 수는 몇 명인가요?",
        "reference_sql": "SELECT COUNT(*) AS total_users FROM users;",
        "solvable": True,
    },
    {
        "difficulty": "기초",
        "question": "완료된 주문 수는 몇 건인가요?",
        "reference_sql": "SELECT COUNT(*) AS completed_orders FROM orders WHERE order_status = 'completed';",
        "solvable": True,
    },
    {
        "difficulty": "기초",
        "question": "메뉴 카테고리 목록을 보여주세요.",
        "reference_sql": "SELECT DISTINCT category FROM menus ORDER BY category;",
        "solvable": True,
    },
    {
        "difficulty": "중간",
        "question": "카테고리별 매출은 어떻게 되나요? completed 주문만 기준으로 알려주세요.",
        "reference_sql": """
            SELECT m.category, ROUND(SUM(oi.line_amount), 0) AS revenue
            FROM orders o
            JOIN order_items oi ON oi.order_id = o.order_id
            JOIN menus m ON m.menu_id = oi.menu_id
            WHERE o.order_status = 'completed'
            GROUP BY m.category
            ORDER BY revenue DESC;
        """,
        "solvable": True,
    },
    {
        "difficulty": "중간",
        "question": "추천 지면에서 메뉴별 CTR 상위 5개를 보여주세요.",
        "reference_sql": """
            WITH imp AS (
                SELECT menu_id, COUNT(*) AS impressions
                FROM view_imp_logs
                WHERE surface = 'recommendation'
                GROUP BY menu_id
            ), clk AS (
                SELECT menu_id, COUNT(*) AS clicks
                FROM click_logs
                WHERE surface = 'recommendation'
                GROUP BY menu_id
            )
            SELECT m.menu_name,
                   i.impressions,
                   COALESCE(c.clicks, 0) AS clicks,
                   ROUND(COALESCE(c.clicks, 0)::numeric / NULLIF(i.impressions, 0), 4) AS ctr
            FROM imp i
            JOIN menus m ON m.menu_id = i.menu_id
            LEFT JOIN clk c ON c.menu_id = i.menu_id
            ORDER BY ctr DESC NULLS LAST, impressions DESC
            LIMIT 5;
        """,
        "solvable": True,
    },
    {
        "difficulty": "중간",
        "question": "최근 30일 completed 주문 기준 상위 메뉴 5개는 무엇인가요?",
        "reference_sql": """
            SELECT m.menu_name,
                   SUM(oi.quantity) AS total_qty,
                   ROUND(SUM(oi.line_amount), 0) AS revenue
            FROM orders o
            JOIN order_items oi ON oi.order_id = o.order_id
            JOIN menus m ON m.menu_id = oi.menu_id
            WHERE o.order_status = 'completed'
              AND o.order_ts >= (SELECT MAX(order_ts) - INTERVAL '30 days' FROM orders)
            GROUP BY m.menu_name
            ORDER BY revenue DESC, total_qty DESC
            LIMIT 5;
        """,
        "solvable": True,
    },
    {
        "difficulty": "고급",
        "question": "추천 지면에서 눈길은 많이 끌었지만 구매로는 잘 안 이어진 메뉴 Top 5는? completed 주문 기준으로 봐주세요.",
        "reference_sql": """
            WITH imp AS (
                SELECT menu_id, COUNT(*) AS impressions
                FROM view_imp_logs
                WHERE surface = 'recommendation'
                GROUP BY menu_id
            ), clk AS (
                SELECT menu_id, COUNT(*) AS clicks
                FROM click_logs
                WHERE surface = 'recommendation'
                GROUP BY menu_id
            ), conv AS (
                SELECT c.menu_id, COUNT(DISTINCT o.order_id) AS converted_orders
                FROM click_logs c
                JOIN orders o
                  ON o.user_id = c.user_id
                 AND o.session_id = c.session_id
                 AND o.order_status = 'completed'
                JOIN order_items oi
                  ON oi.order_id = o.order_id
                 AND oi.menu_id = c.menu_id
                WHERE c.surface = 'recommendation'
                GROUP BY c.menu_id
            )
            SELECT m.menu_name,
                   i.impressions,
                   COALESCE(k.clicks, 0) AS clicks,
                   COALESCE(v.converted_orders, 0) AS converted_orders,
                   ROUND(COALESCE(v.converted_orders, 0)::numeric / NULLIF(k.clicks, 0), 4) AS click_to_order_rate
            FROM imp i
            JOIN menus m ON m.menu_id = i.menu_id
            LEFT JOIN clk k ON k.menu_id = i.menu_id
            LEFT JOIN conv v ON v.menu_id = i.menu_id
            WHERE i.impressions >= 30
              AND COALESCE(k.clicks, 0) > 0
              AND COALESCE(v.converted_orders, 0)::numeric / NULLIF(k.clicks, 0) < 0.05
            ORDER BY i.impressions DESC, click_to_order_rate ASC
            LIMIT 5;
        """,
        "solvable": True,
    },
    {
        "difficulty": "고급",
        "question": "골드 회원이면서 completed 주문이 2건 이상인 재주문 고객들이 가장 많이 산 메뉴 Top 5는 무엇인가요?",
        "reference_sql": """
            WITH loyal_users AS (
                SELECT u.user_id
                FROM users u
                JOIN orders o ON o.user_id = u.user_id
                WHERE u.membership_tier = 'Gold'
                  AND o.order_status = 'completed'
                GROUP BY u.user_id
                HAVING COUNT(DISTINCT o.order_id) >= 2
            )
            SELECT m.menu_name,
                   SUM(oi.quantity) AS total_qty,
                   ROUND(SUM(oi.line_amount), 0) AS revenue
            FROM loyal_users lu
            JOIN orders o ON o.user_id = lu.user_id AND o.order_status = 'completed'
            JOIN order_items oi ON oi.order_id = o.order_id
            JOIN menus m ON m.menu_id = oi.menu_id
            GROUP BY m.menu_name
            ORDER BY total_qty DESC, revenue DESC
            LIMIT 5;
        """,
        "solvable": True,
    },
    {
        "difficulty": "고급",
        "question": "장바구니 이탈률이 높은 메뉴 Top 5는 무엇인가요?",
        "reference_sql": None,
        "solvable": False,
        "reason": "현재 스키마에는 cart 또는 add_to_cart 후 미구매를 판정할 장바구니 테이블이 없습니다.",
    },
    {
        "difficulty": "고급",
        "question": "캠페인별 CAC가 가장 낮은 메뉴는 무엇인가요?",
        "reference_sql": None,
        "solvable": False,
        "reason": "현재 스키마에는 캠페인 비용, 유입 소스, CAC 계산용 마케팅 테이블이 없습니다.",
    },
]


def extract_sql_candidate(resp) -> str:
    metadata_sql = ""
    if hasattr(resp, "metadata") and isinstance(resp.metadata, dict):
        metadata_sql = (resp.metadata.get("sql_query") or "").strip()
        if metadata_sql.lower().startswith(("select", "with")):
            return metadata_sql

    answer = str(getattr(resp, "response", resp)).strip()
    code_blocks = re.findall(r"```sql\s*(.*?)```", answer, re.IGNORECASE | re.DOTALL)
    if code_blocks:
        candidate = code_blocks[0].strip()
        if candidate.lower().startswith(("select", "with")):
            return candidate

    match = re.search(r"((?:WITH|SELECT)\b.*)", answer, re.IGNORECASE | re.DOTALL)
    if match:
        candidate = match.group(1).strip()
        candidate = re.split(r"\n\n|설명:|해석:", candidate, maxsplit=1)[0].strip()
        if candidate.lower().startswith(("select", "with")):
            return candidate

    return metadata_sql or ""


def run_sql_safe(sql: str):
    if not sql.strip():
        return None, "empty_sql"
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text(sql), conn)
        return df, None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:180]}"


def canonicalize(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).lower() for c in out.columns]
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].astype(float).round(4)
        else:
            out[col] = out[col].astype(str)
    out = out.reindex(sorted(out.columns), axis=1)
    if len(out.columns) > 0 and len(out) > 0:
        out = out.sort_values(by=list(out.columns)).reset_index(drop=True)
    return out


def result_matches(sql: str, reference_sql: str) -> tuple[bool, str]:
    got_df, got_err = run_sql_safe(sql)
    if got_err:
        return False, got_err
    ref_df, ref_err = run_sql_safe(reference_sql)
    if ref_err:
        return False, f"reference_error: {ref_err}"

    got_norm = canonicalize(got_df)
    ref_norm = canonicalize(ref_df)
    same = got_norm.equals(ref_norm)
    return same, "exact_match" if same else "result_mismatch"


def ask_engine(label: str, engine_obj, case: dict) -> dict:
    try:
        resp = engine_obj.query(case["question"])
        sql = extract_sql_candidate(resp)
        answer = str(getattr(resp, "response", resp))

        if not case["solvable"]:
            if sql.strip():
                return {
                    "engine": label,
                    "sql": sql,
                    "answer": answer[:220],
                    "success": False,
                    "note": "should_fail_but_generated_sql",
                }
            return {
                "engine": label,
                "sql": sql,
                "answer": answer[:220],
                "success": False,
                "note": "expected_failure",
            }

        matched, note = result_matches(sql, case["reference_sql"])
        return {
            "engine": label,
            "sql": sql,
            "answer": answer[:220],
            "success": matched,
            "note": note,
        }
    except Exception as e:
        return {
            "engine": label,
            "sql": "",
            "answer": f"{type(e).__name__}: {str(e)[:180]}",
            "success": False,
            "note": "exception",
        }


rows = []
for case in QUESTION_CASES:
    print(f"evaluating: {case['question'][:50]}...")
    basic = ask_engine("basic", nlq_basic, case)
    enhanced = ask_engine("enhanced", enhanced_engine, case)
    rows.append(
        {
            "난이도": case["difficulty"],
            "질문": case["question"],
            "정답 가능": case["solvable"],
            "기본 성공": basic["success"],
            "심화 성공": enhanced["success"],
            "기본 엔진 SQL": basic["sql"],
            "심화 엔진 SQL": enhanced["sql"],
            "비고": f"basic={basic['note']}, enhanced={enhanced['note']}",
        }
    )

comparison_df = pd.DataFrame(rows)
comparison_df


evaluating: 전체 사용자 수는 몇 명인가요?...
evaluating: 완료된 주문 수는 몇 건인가요?...
evaluating: 메뉴 카테고리 목록을 보여주세요....
evaluating: 카테고리별 매출은 어떻게 되나요? completed 주문만 기준으로 알려주세요....
evaluating: 추천 지면에서 메뉴별 CTR 상위 5개를 보여주세요....
evaluating: 최근 30일 completed 주문 기준 상위 메뉴 5개는 무엇인가요?...
evaluating: 추천 지면에서 눈길은 많이 끌었지만 구매로는 잘 안 이어진 메뉴 Top 5는? comple...
evaluating: 골드 회원이면서 completed 주문이 2건 이상인 재주문 고객들이 가장 많이 산 메뉴 ...
evaluating: 장바구니 이탈률이 높은 메뉴 Top 5는 무엇인가요?...
evaluating: 캠페인별 CAC가 가장 낮은 메뉴는 무엇인가요?...


,난이도,질문,정답 가능,기본 성공,심화 성공,기본 엔진 SQL,심화 엔진 SQL,비고
0,기초,전체 사용자 수는 몇 명인가요?,True,True,True,SELECT COUNT(user_id) AS total_users FROM users;,SELECT COUNT(DISTINCT user_id) AS total_users ...,"basic=exact_match, enhanced=exact_match"
1,기초,완료된 주문 수는 몇 건인가요?,True,True,False,SELECT COUNT(order_id) AS completed_orders FRO...,SELECT COUNT(*) AS completed_orders_count\nFRO...,"basic=exact_match, enhanced=result_mismatch"
2,기초,메뉴 카테고리 목록을 보여주세요.,True,True,True,SELECT DISTINCT category FROM menus ORDER BY c...,SELECT DISTINCT category\nFROM menus\nORDER BY...,"basic=exact_match, enhanced=exact_match"
3,중간,카테고리별 매출은 어떻게 되나요? completed 주문만 기준으로 알려주세요.,True,False,True,"SELECT m.category, SUM(o.total_amount) AS tota...","SELECT m.category, ROUND(SUM(oi.line_amount), ...","basic=result_mismatch, enhanced=exact_match"
4,중간,추천 지면에서 메뉴별 CTR 상위 5개를 보여주세요.,True,False,True,"SELECT \n m.menu_name, \n COUNT(cl.click...","WITH imp AS (\n SELECT menu_id, COUNT(*) AS...","basic=result_mismatch, enhanced=exact_match"
5,중간,최근 30일 completed 주문 기준 상위 메뉴 5개는 무엇인가요?,True,False,False,"SELECT mi.menu_name, SUM(oi.quantity) AS total...","SELECT m.menu_name,\n SUM(oi.line_amount...","basic=result_mismatch, enhanced=result_mismatch"
6,고급,추천 지면에서 눈길은 많이 끌었지만 구매로는 잘 안 이어진 메뉴 Top 5는? co...,True,False,False,"SELECT m.menu_name, COUNT(v.view_id) AS view_c...",추천 지면에서 눈길은 많이 끌었지만 구매로는 잘 안 이어진 메뉴 Top 5를 찾기 ...,"basic=result_mismatch, enhanced=ProgrammingErr..."
7,고급,골드 회원이면서 completed 주문이 2건 이상인 재주문 고객들이 가장 많이 산...,True,False,False,"SELECT mi.menu_name, SUM(oi.quantity) AS total...",WITH repeat_customers AS (\n SELECT user_id...,"basic=result_mismatch, enhanced=ProgrammingErr..."
8,고급,장바구니 이탈률이 높은 메뉴 Top 5는 무엇인가요?,False,False,False,"SELECT mi.menu_id, m.menu_name, COUNT(cl.click...",장바구니 이탈률은 사용자가 메뉴를 클릭한 후 장바구니에 추가하지 않고 이탈한 비율을...,"basic=should_fail_but_generated_sql, enhanced=..."
9,고급,캠페인별 CAC가 가장 낮은 메뉴는 무엇인가요?,False,False,False,"SELECT m.menu_name, AVG(o.total_amount) AS ave...",캠페인별 CAC(고객 획득 비용)를 계산하기 위해서는 각 메뉴에 대한 클릭 수와 해...,"basic=should_fail_but_generated_sql, enhanced=..."


In [35]:
from IPython.display import display, Markdown
import pandas as pd

# comparison_df 가 문자열 true/false 로 들어와도 안전하게 처리
result_df = comparison_df.copy()

def to_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, str):
        return x.strip().lower() == "true"
    return bool(x)

for col in ["정답 가능", "기본 성공", "심화 성공"]:
    result_df[col] = result_df[col].apply(to_bool)

# 1) 정답 가능한 문제만 따로 계산
solvable_df = result_df[result_df["정답 가능"]].copy()
unsolvable_df = result_df[~result_df["정답 가능"]].copy()

summary_df = pd.DataFrame([
    {
        "engine": "basic",
        "정답 가능 문제 수": len(solvable_df),
        "정답 수": int(solvable_df["기본 성공"].sum()),
        "정답률": round(float(solvable_df["기본 성공"].mean()), 3),
        "정답 불가능 문제 수": len(unsolvable_df),
        "불가능 문제에서 오답 생성 수": int((unsolvable_df["기본 엔진 SQL"].fillna("").str.strip() != "").sum()),
    },
    {
        "engine": "enhanced",
        "정답 가능 문제 수": len(solvable_df),
        "정답 수": int(solvable_df["심화 성공"].sum()),
        "정답률": round(float(solvable_df["심화 성공"].mean()), 3),
        "정답 불가능 문제 수": len(unsolvable_df),
        "불가능 문제에서 오답 생성 수": int((unsolvable_df["심화 엔진 SQL"].fillna("").str.strip() != "").sum()),
    },
])

display(summary_df)
display(Markdown("---"))

# 2) 어떤 문제에서 차이가 났는지 보기
diff_df = result_df[result_df["기본 성공"] != result_df["심화 성공"]].copy()
display(diff_df[["난이도", "질문", "정답 가능", "기본 성공", "심화 성공", "비고"]])
display(Markdown("---"))

# 3) 정답 가능한 문제 중 둘 다 실패한 문제
both_fail_df = solvable_df[(~solvable_df["기본 성공"]) & (~solvable_df["심화 성공"])].copy()
display(both_fail_df[["난이도", "질문", "비고"]])
display(Markdown("---"))

# 4) 간단 해설 자동 생성
basic_acc = float(solvable_df["기본 성공"].mean())
enhanced_acc = float(solvable_df["심화 성공"].mean())

if enhanced_acc > basic_acc:
    verdict = "심화 엔진이 기본 엔진보다 더 높은 정답률을 보였습니다."
elif enhanced_acc < basic_acc:
    verdict = "이번 실험에서는 기본 엔진이 심화 엔진보다 더 높게 나왔습니다."
else:
    verdict = "이번 실험에서는 기본 엔진과 심화 엔진의 정답률이 같았습니다."

markdown_text = f"""
## Text-to-SQL 평가 정리

- 정답 가능한 문제는 **{len(solvable_df)}개**였습니다.
- 기본 엔진 정답률은 **{basic_acc:.1%}** 입니다.
- 심화 엔진 정답률은 **{enhanced_acc:.1%}** 입니다.
- {verdict}

## 해석 포인트

- 기본 엔진은 단순 집계나 단일 테이블 질문에는 비교적 강했습니다.
- 심화 엔진은 `completed 주문만 집계`, `추천 지면`, `CTR 계산`처럼 **도메인 규칙이 필요한 질문**에서 더 유리해야 합니다.
- 다만 이번 결과처럼 심화 엔진도 설명문을 섞어 내거나, SQL은 맞지만 **기준 결과와 다르게 집계**하면 실패로 잡힐 수 있습니다.
- `장바구니 이탈률`, `캠페인별 CAC` 같은 질문은 현재 스키마에 필요한 테이블이 없어서 **의도적으로 풀 수 없게 만든 문제**입니다.

## 추가로 볼 것

- `기본 성공 != 심화 성공` 인 문제:
  심화가 실제로 도움된 질문이 무엇인지 확인합니다.
- `둘 다 실패` 한 문제:
  프롬프트 보강만으로 해결 가능한지, 아니면 스키마 자체가 더 필요한지 구분합니다.
"""

display(Markdown(markdown_text))

,engine,정답 가능 문제 수,정답 수,정답률,정답 불가능 문제 수,불가능 문제에서 오답 생성 수
0,basic,8,3,0.375,2,2
1,enhanced,8,4,0.500,2,2


---

,난이도,질문,정답 가능,기본 성공,심화 성공,비고
1,기초,완료된 주문 수는 몇 건인가요?,True,True,False,"basic=exact_match, enhanced=result_mismatch"
3,중간,카테고리별 매출은 어떻게 되나요? completed 주문만 기준으로 알려주세요.,True,False,True,"basic=result_mismatch, enhanced=exact_match"
4,중간,추천 지면에서 메뉴별 CTR 상위 5개를 보여주세요.,True,False,True,"basic=result_mismatch, enhanced=exact_match"


---

,난이도,질문,비고
5,중간,최근 30일 completed 주문 기준 상위 메뉴 5개는 무엇인가요?,"basic=result_mismatch, enhanced=result_mismatch"
6,고급,추천 지면에서 눈길은 많이 끌었지만 구매로는 잘 안 이어진 메뉴 Top 5는? co...,"basic=result_mismatch, enhanced=ProgrammingErr..."
7,고급,골드 회원이면서 completed 주문이 2건 이상인 재주문 고객들이 가장 많이 산...,"basic=result_mismatch, enhanced=ProgrammingErr..."


---


## Text-to-SQL 평가 정리

- 정답 가능한 문제는 **8개**였습니다.
- 기본 엔진 정답률은 **37.5%** 입니다.
- 심화 엔진 정답률은 **50.0%** 입니다.
- 심화 엔진이 기본 엔진보다 더 높은 정답률을 보였습니다.

## 해석 포인트

- 기본 엔진은 단순 집계나 단일 테이블 질문에는 비교적 강했습니다.
- 심화 엔진은 `completed 주문만 집계`, `추천 지면`, `CTR 계산`처럼 **도메인 규칙이 필요한 질문**에서 더 유리해야 합니다.
- 다만 이번 결과처럼 심화 엔진도 설명문을 섞어 내거나, SQL은 맞지만 **기준 결과와 다르게 집계**하면 실패로 잡힐 수 있습니다.
- `장바구니 이탈률`, `캠페인별 CAC` 같은 질문은 현재 스키마에 필요한 테이블이 없어서 **의도적으로 풀 수 없게 만든 문제**입니다.

## 추가로 볼 것

- `기본 성공 != 심화 성공` 인 문제:
  심화가 실제로 도움된 질문이 무엇인지 확인합니다.
- `둘 다 실패` 한 문제:
  프롬프트 보강만으로 해결 가능한지, 아니면 스키마 자체가 더 필요한지 구분합니다.
